# 03 — Entrenamiento de modelos

Compara Logistic Regression, Decision Tree y Random Forest sobre el dataset agregado por tiro, usando validación cruzada 5-fold, y ajusta hiperparámetros del mejor modelo. Reutiliza las funciones de `scripts/train_model.py` para no duplicar lógica.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from scripts.train_model import (
    load_dataset, build_models, cross_validate_models, tune_best_model,
    optimal_threshold, evaluate, RANDOM_STATE,
)
from sklearn.model_selection import train_test_split

In [2]:
X, y, feature_cols = load_dataset(Path('../data/processed/features_por_tiro.csv'))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'{len(X_train)} train / {len(X_test)} test tiros, {len(feature_cols)} features')

328 train / 83 test tiros, 57 features


## Baseline models (5-fold CV, class_weight balanceado)

In [3]:
models = build_models('class_weight')
cv_results = cross_validate_models(models, X_train, y_train)
cv_results

,modelo,accuracy,precision,recall,f1
0,logistic_regression,0.557995,0.389727,0.523715,0.444600
1,random_forest,0.609883,0.419336,0.386957,0.399242
2,decision_tree,0.567273,0.377553,0.360474,0.361929


In [4]:
best_name = cv_results.iloc[0]['modelo']
print('Mejor modelo por F1 en CV:', best_name)

Mejor modelo por F1 en CV: logistic_regression


## Hyperparameter tuning (GridSearchCV, scoring=F1)

In [5]:
best_model, best_params, best_cv_f1 = tune_best_model(best_name, models, X_train, y_train)
print('Mejores hiperparametros:', best_params)
print(f'F1 en CV tras tuning: {best_cv_f1:.3f}')

Mejores hiperparametros: {'model__C': 0.1, 'model__penalty': 'l2'}
F1 en CV tras tuning: 0.446


## Umbral óptimo (maximiza F1 en el set de entrenamiento)

In [6]:
best_model.fit(X_train, y_train)
y_proba_train = best_model.predict_proba(X_train)[:, 1]
thr = optimal_threshold(y_train, y_proba_train)
print(f'Umbral optimo: {thr:.2f}')

Umbral optimo: 0.41


## Evaluación en el set de test

In [7]:
test_metrics = evaluate(best_model, X_test, y_test, thr)
{k: v for k, v in test_metrics.items() if k not in ('y_proba','y_pred')}

{'threshold': 0.4149999999999999,
 'accuracy': 0.46987951807228917,
 'precision': 0.36666666666666664,
 'recall': 0.7857142857142857,
 'f1': 0.5,
 'roc_auc': 0.6012987012987012,
 'confusion_matrix': [[17, 38], [6, 22]]}

Nota: `scripts/train_model.py` guarda el modelo final (`models/best_model.pkl`), las métricas (`results/metrics.json`) y los gráficos de evaluación. El notebook 04 los carga y analiza en detalle.